# Lab 04: 多層感知器 (MLP) — 分類手寫數字

在本單元中，我們將學習如何將多個神經元組成「深度」神經網路。我們將使用著名的 **MNIST 手寫數字資料集**，訓練一個多層感知器 (MLP) 模型來識別 0 到 9 的手寫數字影像！

### GPU 環境測試
在開始之前，請先確認您的執行環境已成功啟用 GPU 加速器（例如 Google Colab 中的 T4 GPU）。

**【重要】**若下方輸出顯示為 CPU，請點選右上角『連線 / 變更執行階段類型』，將硬體加速器切換為 GPU 後再往下執行，以防切換執行階段時已安裝的套件與下載的資料被清空重設！

In [ ]:
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'GPU 裝置: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (建議切換 T4 GPU)"}')


### 環境安全網與資料準備
確認已啟用 GPU 後，接著我們安裝缺少的套件，並測試 MNIST 資料集連線是否正常。

In [ ]:
import subprocess, sys

def ensure_pkg(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'torchvision', 'matplotlib']:
    ensure_pkg(pkg)

# 測試 MNIST 下載 (用少量資料測試)
try:
    import torchvision
    test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True,
                                          transform=torchvision.transforms.ToTensor())
    print(f'MNIST 測試集: {len(test_ds)} 張圖片已就緒')
except Exception as e:
    print(f'[警告] MNIST 下載失敗: {e}')
    print('[備用] 將使用隨機張量模擬 MNIST 圖片資料')
    print('   模擬資料不影響學習模型架構的核心概念')


### 任務 1: 載入並觀察手寫數字資料集 (MNIST)
我們使用 `torchvision` 套件下載並載入 MNIST 影像。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 定義影像的前處理：轉為張量並標準化 (Normalize)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 下載並讀取訓練集與測試集
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 建立 DataLoader (批次讀取資料器，每批讀取 64 張圖片)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# 顯示一小批訓練圖片來看看
images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')
plt.show()

### 任務 2: 定義多層感知器 (MLP)
由於 MNIST 影像大小是 $28 \times 28$ 像素，我們要先用資料展平 (Flatten) `nn.Flatten()` 將其展開成 $784$ 維的一維向量，然後建立含有隱藏層與 ReLU 活化函數的模型。

**MLP 架構**：
輸入層 ($784$ 維) → 隱藏層 1 ($128$ 維) → ReLU → 隱藏層 2 ($64$ 維) → ReLU → 輸出層 ($10$ 個分類，代表 0~9)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        # TODO: 補全 MLP 的架構定義
        # 提示：用 nn.Sequential 依序堆疊下列各層
        #   nn.Flatten()          # 28x28 -> 784
        #   nn.Linear(784, 128)   # 隱藏層 1
        #   nn.ReLU()
        #   nn.Linear(128, 64)    # 隱藏層 2
        #   nn.ReLU()
        #   nn.Linear(64, 10)     # 輸出層 (10個類別)
        # 並把整個 nn.Sequential 指定給 self.network
        # ??? (請在此填寫你的答案)

    def forward(self, x):
        # (已幫你完成) 將輸入 x 通過網路
        out = self.network(x)
        return out

model = MLP()
print(model)

### 任務 3: 訓練與評估函數
我們使用 **交叉熵誤差評分函數 (Loss Function) `nn.CrossEntropyLoss`** (適合多分類) 與 **Adam 智慧調整器 (Optimizer)**。

In [ ]:
# 設定運算裝置 (GPU 優先)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        # 將資料移動到 GPU / CPU
        images, labels = images.to(device), labels.to(device)
        
        # 模型預測計算 (前向傳播)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # (已幫你完成) 優化三步驟：梯度歸零 → 誤差修正計算 (反向傳播) → 更新參數
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad(): # 驗證階段不計算梯度，加快速度
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    val_loss = running_loss / len(loader.dataset)
    val_acc = (correct / total) * 100
    return val_loss, val_acc

### 任務 4: 執行訓練
我們訓練 5 個 Epoch (輪)，觀察訓練集與驗證集的 Loss 和 Accuracy 變化。

In [ ]:
epochs = 5
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch [{epoch+1}/{epochs}] ")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

### 任務 5: 視覺化訓練過程
我們繪製 Loss 與 Accuracy 曲線，確認模型是否在正常學習。

In [ ]:
plt.figure(figsize=(12, 4))

# 繪製 Loss 曲線
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 繪製 Accuracy 曲線
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.title('Accuracy History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.show()

### 任務 6: 實際預測與展示
我們從測試集隨機抽出幾張圖片，讓剛訓練好的 MLP 模型進行預測，並印出預測結果。

In [ ]:
model.eval()
# 建立一個 shuffle=True 的臨時 DataLoader 來隨機抽取 10 張圖片
temp_loader = torch.utils.data.DataLoader(test_dataset, batch_size=10, shuffle=True)
images, labels = next(iter(temp_loader))
images_dev = images.to(device)

with torch.no_grad():
    outputs = model(images_dev)
    _, preds = outputs.max(1)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    color = 'green' if preds[i].item() == labels[i].item() else 'red'
    plt.title(f"Pred: {preds[i].item()}\nTrue: {labels[i].item()}", color=color)
    plt.axis('off')
plt.show()

---

## Lab 04 學習重點小結
### 核心概念掌握
- MLP (多層感知器) 透過多個隱藏層堆疊，學習更複雜的非線性特徵。
- 資料展平 (Flatten) `nn.Flatten()` 將 28×28 的二維影像展平為 784 維一維向量。
- 交叉熵損失 (CrossEntropyLoss) 適用於多分類問題的損失計算。
- 過擬合：模型在訓練集表現很好，在測試集卻變差，需用 Dropout 等技術防止。

### 快速自評測驗

**Q1. MNIST 影像的輸入維度（經過 Flatten 展平後）為？**
- A. 28
- B. 784
- C. 1024

<details><summary>查看解答</summary>

**答案：B — 28 × 28 = 784 個像素值組成一維輸入向量。**

</details>

**Q2. 為何 MLP 可以解決 XOR 問題但單層感知器不行？**
- A. MLP 參數量更多
- B. 多層結構引入了非線性特徵組合
- C. MLP 使用了 GPU

<details><summary>查看解答</summary>

**答案：B — 多層結構加上非線性活化函數可以學習任意複雜的決策邊界。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「MLP (多層感知器) 透過多個隱藏層堆疊，學習更複雜的非線」
- [ ] 我可以用一句話解釋「資料展平 (Flatten) 將 28×28 的二維影像展平」
- [ ] 我可以用一句話解釋「交叉熵損失 (CrossEntropyLoss) 適用於多分」
- [ ] 我的程式碼成功執行並得到預期輸出